# D2 — Ministral 3 3B: baseline vs retrieval + tool use (Colab)

Corre en la GPU gratuita de Colab (respaldo declarado en el D1) el **baseline** del D1 (prompt directo) y las variantes con retrieval + tool use, sobre la misma transcripción y con el mismo corrector automático.

1. `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.
2. `Entorno de ejecución → Ejecutar todas`. La primera celda clona la rama `exp/rag-tool-use` desde GitHub (con `USAR_GITHUB = False` pide el zip del experimento).

Usa `ministral-3:3b` local (**nunca** `ministral-3:3b-cloud`), `num_ctx` 8192, temperatura 0 y seed 42. El baseline y la solución corren en el mismo hardware.

In [ ]:
USAR_GITHUB = True             # False: pide subir el zip del experimento en vez de clonar
REPO_URL = "https://github.com/benjaminnalvear-dev/genai-vio-meeting-minutes.git"
BRANCH = "exp/rag-tool-use"
OLLAMA_VERSION = "0.33.3"      # misma versión usada en las corridas de desarrollo
MODEL = "ministral-3:3b"
EXPECTED_DIGEST = "f04aa1c738f6"  # digest del modelo en el D1

%cd /content
!rm -rf genai-vio-meeting-minutes
if USAR_GITHUB:
    !git clone -q -b $BRANCH $REPO_URL
else:
    from google.colab import files
    subido = files.upload()          # elegir d2_rag_tool_use.zip
    nombre = next(iter(subido))
    !unzip -q -o "$nombre"
%cd /content/genai-vio-meeting-minutes/experimentos/rag_tool_use

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!sudo apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=$OLLAMA_VERSION sh

In [ ]:
import subprocess, time, urllib.request, json
server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://localhost:11434/api/version").read().decode()); break
    except Exception:
        time.sleep(1)
assert not MODEL.endswith("-cloud"), "El modelo debe correr localmente"
!ollama pull $MODEL
tags = json.loads(urllib.request.urlopen("http://localhost:11434/api/tags").read())
digest = next(m["digest"] for m in tags["models"] if m["name"] == MODEL)
print("digest:", digest[:12], "| coincide con el D1:", digest.startswith(EXPECTED_DIGEST))

In [ ]:
# Pruebas del resolvedor temporal y chequeo del corrector sobre la salida guardada del D1
!python -m unittest discover -s tests -v 2>&1 | tail -5
!python run.py score ../../pruebas/01_salida_ministral_8k.md | head -20

In [ ]:
# Baseline + las seis variantes (S1–S4, S3b y el control) y comparación. En una T4 tarda ~10 minutos.
!python run.py todo

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
folder = sorted(Path("resultados").glob("*/comparacion.md"), key=lambda p: p.stat().st_mtime)[-1]
display(Markdown(folder.read_text()))

In [ ]:
# Descargar resultados (prompts, respuestas crudas, métricas) para subirlos al repo
!zip -qr /content/resultados_colab.zip resultados
from google.colab import files
files.download("/content/resultados_colab.zip")

## Probar con otra reunión

Sube una transcripción (formato canónico, `[09:30] Nombre: texto` o `Nombre: texto`; la cabecera debería traer `Fecha:`, `Participantes:` y `Ausentes:`). El modo `auto` usa S2 si la reunión cabe en el contexto y activa el RAG si no.

In [ ]:
from google.colab import files
subida = files.upload()                 # elegir la transcripción .md o .txt
ruta = next(iter(subida))
!python run.py acta "$ruta" --config auto
# Si la transcripción no trae fecha o participantes:
# !python run.py acta "$ruta" --fecha 2026-10-01 --presentes "Ana Díaz, Luis Mora" --ausentes "Marta"
